# Prepare the Data

## Import Libraries and Load Data

In [14]:
import pickle

import numpy as np
import pandas as pd

from preparation import build_preprocessor

train_set = pd.read_parquet("train_set.parquet")
test_set = pd.read_parquet("test_set.parquet")

y_train = train_set["early_spring"]

## Data Cleaning

1. **Fix or remove outliers:** Does not apply. The extreme precipitation, snowfall, and snow depth values are real storms, not measurement errors, so they are kept. The `log1p` transform reduces their influence instead.
2. **Fill in missing values:** Missing values are imputed with the median *after* daily values are aggregated. A year with a few missing days still gets a real monthly mean from the days that were observed, so the imputer only has to fill whole months that are missing (January 2001 and February 2004 air temperatures) and the climate indices before 1950.
3. **Drop rows or columns:** The February 29 columns are not dropped and are no longer separate features. February 29 is treated like any other day: it counts toward February, and each of the 10 days before March 1 is lined up by its distance from March 1 (February 19–28, or February 20–29 in leap years). Every year gets the same columns with no leap day NaNs. The March 1 columns are dropped so the February summaries only use February days, matching the monthly features in the Exploration notebook. Otherwise a year missing all of February, like 2004, would get its February mean from March 1 alone. No rows are dropped, because every year is needed with only 57 training observations.

## Feature Selection

The three Niño 3.4 months are averaged into one feature because they are almost perfectly correlated (r = 0.92 to 0.97). `year` and `phil_pred` are kept by default but can be dropped as hyperparameters. 

## Feature Engineering

- **Aggregate features:** Daily values are summarized per month: means for temperature, snow depth, and soil; totals for precipitation and snowfall.
- **Add promising features:** Average temperature standard deviation, minimum, and maximum per month; percent of days above freezing; and percent of snowy days. Percentages are calculated out of the days that were actually observed, which fixes the Exploration finding that 2001 was counted as having 0 days above freezing.
- **Last 10 days features:** Each of the 10 days before March 1 keeps its own daily values (for example, `avg_temp_mar1_minus01` is the day before March 1), on top of the full January and February summaries. The days are counted back from March 1, so they are February 19–28 in normal years and February 20–29 in leap years, and every year has the same features. They overlap the February window on purpose.
- **Transform features:** `log1p` is applied to precipitation, snowfall, and snow depth because they are zero-inflated and right-skewed.
- **Discretize continuous features / decompose features:** Does not apply. There are no dates or categorical features left to decompose after aggregation.

## Feature Scaling

All features are standardized with `StandardScaler`. This puts soil temperature (°C), air temperature (°F), soil moisture (about 0.47), and the climate indices on the same scale, so no unit conversion is needed.

## Define the Preprocessing Transformers

The custom transformers (`DropColumns`, `WeatherAggregator`, `ClimateFeatures`, `YearPhil`), their helper functions, and `build_preprocessor()` are defined in `preparation.py`. Because they live in a module instead of this notebook, `preprocessor.pkl` refers to them as `preparation.DropColumns` and so on, so any notebook (and the final `model.pkl`) can load it as long as `preparation.py` is in the same folder.

## Build and Fit the Preprocessor

In [15]:
preprocessor = build_preprocessor()

X_train = preprocessor.fit_transform(train_set)

print("Training shape:", X_train.shape)
print("Missing values:", X_train.isna().sum().sum())

X_train.head()

Training shape: (57, 160)
Missing values: 0


,precipitation_jan_total,precipitation_feb_total,snowfall_jan_total,snowfall_feb_total,snowdepth_jan_mean,snowdepth_feb_mean,precipitation_mar1_minus10,precipitation_mar1_minus09,precipitation_mar1_minus08,precipitation_mar1_minus07,...,max_soil_moisture_mar1_minus09,max_soil_moisture_mar1_minus08,max_soil_moisture_mar1_minus07,max_soil_moisture_mar1_minus06,max_soil_moisture_mar1_minus05,max_soil_moisture_mar1_minus04,max_soil_moisture_mar1_minus03,max_soil_moisture_mar1_minus02,max_soil_moisture_mar1_minus01,nino34
0,-1.248973,-0.025729,-0.071068,0.862266,0.241708,0.496979,-0.664050,2.187411,-0.053542,-0.592907,...,-0.712012,-0.970624,-1.033316,-1.106563,-1.138493,-1.145944,-1.277337,-1.268993,-1.141033,-0.305348
1,0.978692,0.772779,0.904019,-0.265668,1.093221,-0.099929,1.837392,0.688004,-0.053542,-0.403107,...,1.471733,-0.221716,-0.288868,0.318840,0.262424,-0.282299,-0.438653,-0.302103,-0.376869,-0.010960
2,0.687879,-0.025729,1.320016,-0.448312,0.387007,-0.004996,0.141389,-0.473983,-0.257973,-0.592907,...,-0.217579,1.463326,-1.909138,-1.730178,-1.680783,-1.873224,0.085524,-1.038781,-1.480662,-1.072127
3,0.681660,-1.059277,-1.269920,0.967694,-1.372062,-0.333436,-0.664050,-0.473983,-0.524752,-0.592907,...,-0.423593,-0.689784,-0.770570,-0.794756,-0.731775,0.081340,-0.019312,-0.578357,-0.631590,-0.757200
4,0.027989,-0.428228,-0.811097,-2.217677,-0.982753,-1.195480,-0.664050,0.042758,0.143521,-0.592907,...,1.512936,1.556940,-0.288868,-0.572037,-0.641393,-0.873214,-0.333818,-0.025848,0.005214,0.177312


## Check the Engineered Features Before Imputation

In [16]:
before_imputation = preprocessor[:4].transform(train_set)

before_imputation.loc[
    train_set["year"].isin([2001, 2004]),
    ["avg_temp_jan_mean", "above_freezing_jan_pct", "avg_temp_feb_mean", "above_freezing_feb_pct", "avg_temp_mar1_minus01"],
].assign(year=train_set["year"])

,avg_temp_jan_mean,above_freezing_jan_pct,avg_temp_feb_mean,above_freezing_feb_pct,avg_temp_mar1_minus01,year
7,20.583333,0.166667,NaN,NaN,NaN,2004
10,NaN,NaN,35.942308,0.615385,32.0,2001


2001 (no January air temperatures) and 2004 (no February air temperatures) now show missing values for those months (and 2004 for each of its last 10 days) instead of a false 0% (or, for 2004, a February mean built from March 1 alone). The imputer fills them in, so the model never sees an incorrect "always frozen" month.

## Check the Preprocessor on New Data

In [19]:
X_test = preprocessor.transform(test_set)

unlabeled = test_set.drop(columns="early_spring")
X_unlabeled = preprocessor.transform(unlabeled)

missing_climate = unlabeled.head(1).copy()
climate_columns = [col for col in missing_climate.columns if col.startswith(("nino34", "nao"))]
missing_climate[climate_columns] = np.nan
X_missing_climate = preprocessor.transform(missing_climate)

print("Test shape:", X_test.shape)
print("Unlabeled shape:", X_unlabeled.shape)
print("Missing values with no climate data:", X_missing_climate.isna().sum().sum())

Test shape: (15, 160)
Unlabeled shape: (15, 160)
Missing values with no climate data: 0


The preprocessor handles the test set, data without the `early_spring` column (like the output of `generate_samples()`), and years where the climate indices are not available yet.

## Save the Preprocessor

In [20]:
with open("preprocessor.pkl", "wb") as file:
    pickle.dump(preprocessor, file)

with open("preprocessor.pkl", "rb") as file:
    loaded = pickle.load(file)

print("Loaded preprocessor matches:", np.allclose(loaded.transform(test_set), X_test, equal_nan=True))

Loaded preprocessor matches: True


## Hyperparameters for Fine-Tuning


- `climate__collapse_nino`: `True`, `False`
- `year_phil__include_year`: `True`, `False`
- `year_phil__include_phil`: `True`, `False`
- `columns__skewed__impute__strategy`, `columns__remainder__strategy`: `"median"`, `"mean"`
- `columns__skewed__log`: `FunctionTransformer(np.log1p)`, `"passthrough"`
- `reduce`: `"passthrough"`, `SelectKBest(k=...)`, `PCA(n_components=...)`